# 🍿 Movie Night Picker
Pick movies you love → get recommendations, **including what's in theaters / trending right now**, for one person or two (group mode).

**How it works:** every movie (title + genres + plot + keywords) becomes a vector with a small pre-trained text model. Your taste = the average of the vectors of your picks, and recommendations are the nearest movies. No training loop, so brand-new releases work instantly.

**Before you run (one-time setup)**
1. Left sidebar → 🔑 **Secrets** → *Add new secret* → name: `TMDB_KEY`, value: your TMDB key (the API Key *or* the long Read Access Token, both work) → turn on *Notebook access*.
2. *(Recommended)* Runtime → Change runtime type → **T4 GPU**. It makes the embedding step ~1 minute instead of ~8.
3. Run the cells top to bottom (Runtime → Run all works too). If a step fails, the error tells you what to fix, and you can paste it to me.

## 0 · Setup

In [ ]:
!pip -q install sentence-transformers gradio kagglehub pyarrow huggingface_hub

In [ ]:
import os, requests
from google.colab import userdata

# ---- settings you can tweak ----
MAX_MOVIES = 20000   # catalog size (smaller = faster embedding)
MIN_VOTES = 40       # skip movies that almost nobody rated

try:
    os.environ["TMDB_KEY"] = userdata.get("TMDB_KEY")
except Exception:
    raise RuntimeError("Secret TMDB_KEY not found. Sidebar 🔑 Secrets -> Add new secret -> name TMDB_KEY, "
                       "and switch on 'Notebook access'. Then re-run this cell.")

# quick check that the key works
key = os.environ["TMDB_KEY"].strip()
headers, params = {"accept": "application/json"}, {}
if key.startswith("eyJ") or len(key) > 40:
    headers["Authorization"] = f"Bearer {key}"
else:
    params["api_key"] = key
r = requests.get("https://api.themoviedb.org/3/configuration", headers=headers, params=params, timeout=15)
print("TMDB key works ✅" if r.ok else f"❌ TMDB said {r.status_code}: {r.text[:200]}")

## 1 · Get the movie catalog (Kaggle TMDB dataset)
~1M movies from TMDB. We keep the well-known ones (enough votes, has plot + poster). The newest releases come from the live API later.

In [ ]:
import glob, kagglehub

SLUG = "asaniczka/tmdb-movies-dataset-2023-930k-movies"
try:
    path = kagglehub.dataset_download(SLUG)
except Exception as e:
    print("Anonymous download failed (", type(e).__name__, ") -> logging in to Kaggle...")
    kagglehub.login()          # paste your Kaggle username + API token when asked
    path = kagglehub.dataset_download(SLUG)

csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
csv_path = max(csvs, key=os.path.getsize)
print(csv_path, "-", round(os.path.getsize(csv_path) / 1e6), "MB")

In [ ]:
import numpy as np, pandas as pd

WANT = ["id", "title", "status", "release_date", "vote_average", "vote_count", "runtime", "adult",
        "overview", "tagline", "popularity", "poster_path", "genres", "keywords", "original_language"]
raw = pd.read_csv(csv_path, usecols=lambda c: c in WANT, low_memory=False)
print("raw:", raw.shape, "| missing columns:", [c for c in WANT if c not in raw.columns])

df = raw.copy()
for c in ["tagline", "genres", "keywords", "original_language"]:
    if c not in df: df[c] = ""
for c in ["runtime", "popularity", "vote_average", "vote_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce") if c in df else 0
if "adult" in df:  df = df[~df["adult"].astype(str).str.lower().eq("true")]
if "status" in df: df = df[df["status"].eq("Released")]

df = df.dropna(subset=["title", "overview", "poster_path", "release_date"])
df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df = df.dropna(subset=["year"])
df["vote_count"] = df["vote_count"].fillna(0)
df = df[(df["overview"].str.len() >= 40) & (df["vote_count"] >= MIN_VOTES)]
df = df.drop_duplicates(subset=["id"])

# sorted by popularity: row order must never change after embeddings are built
df = df.sort_values("vote_count", ascending=False).head(MAX_MOVIES).reset_index(drop=True)
for c in ["tagline", "genres", "keywords", "original_language"]:
    df[c] = df[c].fillna("").astype(str)
for c in ["runtime", "popularity", "vote_average"]:
    df[c] = df[c].fillna(0)
df["year"] = df["year"].astype(int)

keep = ["id", "title", "year", "genres", "keywords", "overview", "tagline", "poster_path",
        "vote_average", "vote_count", "popularity", "runtime", "original_language"]
df[keep].to_parquet("catalog.parquet")

print(f"catalog: {len(df):,} movies | newest release date in dataset: {pd.to_datetime(df['release_date']).max().date()}")
df[["title", "year", "genres", "vote_count"]].head(8)

## 2 · Turn every movie into a vector
Model: `all-MiniLM-L6-v2` (small, fast, good for semantic similarity). Each movie's text = title + genres + plot + tagline + keywords.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

def build_text(r):
    parts = [r.title]
    if r.genres:   parts.append("Genres: " + r.genres)
    parts.append(r.overview)
    if r.tagline:  parts.append(r.tagline)
    if r.keywords: parts.append("Keywords: " + ", ".join(k.strip() for k in r.keywords.split(",")[:15]))
    return ". ".join(parts)

texts = [build_text(r) for r in df.itertuples()]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| movies:", len(texts))

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
emb = model.encode(texts, batch_size=128, show_progress_bar=True,
                   normalize_embeddings=True, convert_to_numpy=True)
np.save("embeddings.npy", emb.astype("float16"))
print("embeddings:", emb.shape)

## 3 · The recommender engine (`recsys.py`)
Each candidate is scored against **each of your picks** (not one blurred average) plus a genre bonus, then results are chosen to be diverse and spread across your picks. A small "crowd-pleaser" popularity boost and filters come on top, and every result gets a "why this?" line. Also fetches live in-theaters / trending movies from TMDB.

In [ ]:
%%writefile recsys.py
"""Core recommender logic for Movie Night Picker.

Files expected next to this module:
  catalog.parquet  - movie table, sorted by vote_count (most popular first)
  embeddings.npy   - one normalized text embedding per catalog row
"""
import os
import re
import time
import html

import numpy as np
import pandas as pd
import requests

MODEL_NAME = "all-MiniLM-L6-v2"
TMDB = "https://api.themoviedb.org/3"
IMG = "https://image.tmdb.org/t/p/w342"
HERE = os.path.dirname(os.path.abspath(__file__))

cat = pd.read_parquet(os.path.join(HERE, "catalog.parquet")).reset_index(drop=True)
emb = np.load(os.path.join(HERE, "embeddings.npy")).astype("float32")
assert len(cat) == len(emb), "catalog and embeddings are out of sync - rebuild both"

cat["year"] = cat["year"].astype(int)
cat["label"] = cat["title"] + " (" + cat["year"].astype(str) + ")"

_p = np.log1p(cat["vote_count"].values.astype("float32"))
pop = (_p - _p.min()) / (_p.max() - _p.min() + 1e-9)  # 0..1 "how well known"

_label_to_idx = {}
for _i, _l in enumerate(cat["label"]):
    _label_to_idx.setdefault(_l, _i)  # first hit = most popular duplicate

_gsets = [{g.strip() for g in str(s).split(",") if g.strip()} for s in cat["genres"]]
_JUNK = ("credits", "stinger", "sequel", "woman director")   # tags that say nothing about the story


def _clean_kw(s):
    return {k.strip().lower() for k in str(s).split(",")
            if k.strip() and not any(j in k.lower() for j in _JUNK)}


_ksets = [_clean_kw(s) for s in cat["keywords"]]
ALL_GENRES = sorted(set().union(*_gsets))
_gidx = {g: i for i, g in enumerate(ALL_GENRES)}


def _multihot(genre_strings):
    M = np.zeros((len(genre_strings), len(ALL_GENRES)), dtype="float32")
    for r, s in enumerate(genre_strings):
        for g in str(s).split(","):
            g = g.strip()
            if g in _gidx:
                M[r, _gidx[g]] = 1.0
    return M


_G = _multihot(cat["genres"])


_STOP = {"the", "a", "an", "of", "and", "in", "to", "at", "on", "for", "with"}
_ROMAN = {"ii", "iii", "iv", "vi", "vii", "viii", "ix"}


def franchise_key(title):
    """Cheap 'same series' key: 'Avengers: Endgame', 'The Avengers' and 'Avengers 2' all -> 'avengers';
    'Pirates of the Caribbean: Dead Man's Chest' -> 'pirates caribbean'."""
    t = re.split(r"\s*:\s*|\s+[-\u2013\u2014]\s+", str(title).lower())[0]
    t = t.replace("'", "").replace("\u2019", "")
    toks = [w for w in re.findall(r"[a-z0-9\-]+", t)
            if w not in _STOP and not w.isdigit() and w not in _ROMAN]
    return " ".join(toks[:2]) if toks else str(title).lower()


cat["fkey"] = cat["title"].map(franchise_key)
_fkeys = cat["fkey"].values


def esc(s):
    return html.escape(str(s))


def picker_choices(n=6000):
    """Labels shown in the 'pick your favourites' dropdown (most popular first)."""
    return cat["label"].head(n).tolist()


def find(query, n=10):
    """Search the catalog by title, e.g. find('inception')."""
    q = query.lower()
    m = cat[cat["title"].str.lower().str.contains(q, regex=False)]
    return m["label"].head(n).tolist()


def _unit(v):
    return v / (np.linalg.norm(v) + 1e-9)


def _combine(sims):
    """One person -> their similarity. Two people -> mix of average and the weaker side,
    so a pick has to work for BOTH of them."""
    if len(sims) == 1:
        return sims[0]
    a, b = sims[0], sims[1]
    return 0.5 * (a + b) + 0.5 * np.minimum(a, b)


def _person_scores(E, G, g, genre_w=0.2):
    """Score every candidate row (embeddings E, genre multi-hot G) for one person whose
    picks are the catalog rows `g`.
    - 0.5 * average match + 0.5 * best single match: a candidate only has to fit ONE of your
      picks, so mixed tastes (Joker + John Wick + a cartoon) don't get blurred into a blob.
    - genre bonus: candidates whose genres appear in your picks score higher, so a cartoon
      doesn't sneak in just because its plot mentions a villain."""
    sims = E @ emb[g].T
    s = 0.5 * sims.mean(1) + 0.5 * sims.max(1)
    if genre_w:
        w = _G[g].mean(0)                                   # share of picks having each genre
        s = s + genre_w * (G @ w) / np.maximum(G.sum(1), 1.0)
    return s


def _select(scores, pick_idx, n, lam=0.15, quota_pen=0.04, pool=100):
    """Greedy pick of n results from the top `pool`, trading score against
    (a) being too similar to results already chosen and
    (b) over-using the same liked movie as the reason."""
    cand = [int(i) for i in np.argsort(-scores)[:pool] if np.isfinite(scores[i])]
    if not cand:
        return []
    near = dict(zip(cand, np.argmax(emb[cand] @ emb[pick_idx].T, axis=1)))
    used, chosen, remaining = {}, [], list(cand)
    while remaining and len(chosen) < n:
        def adj(i):
            v = scores[i] - quota_pen * used.get(near[i], 0)
            if chosen:
                v -= lam * float((emb[chosen] @ emb[i]).max())
            return v
        best = max(remaining, key=adj)
        chosen.append(best)
        remaining.remove(best)
        used[near[best]] = used.get(near[best], 0) + 1
    return chosen


# ---------------------------------------------------------------- live TMDB data
_model = None


def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer(MODEL_NAME)
    return _model


def _tmdb_get(path, **params):
    key = os.environ.get("TMDB_KEY", "").strip()
    if not key:
        raise RuntimeError("TMDB_KEY is not set")
    headers, query = {"accept": "application/json"}, dict(params)
    if key.startswith("eyJ") or len(key) > 40:      # v4 read-access token
        headers["Authorization"] = f"Bearer {key}"
    else:                                           # v3 api key
        query["api_key"] = key
    r = requests.get(TMDB + path, headers=headers, params=query, timeout=15)
    r.raise_for_status()
    return r.json()


def _year(s):
    try:
        return int(str(s)[:4])
    except Exception:
        return 0


_live = {"df": None, "emb": None, "t": 0.0, "error": None}


def get_live(ttl=6 * 3600):
    """Movies that are in theaters / trending right now (cached for a few hours)."""
    if _live["df"] is not None and time.time() - _live["t"] < ttl:
        return _live["df"], _live["emb"]
    try:
        gmap = {g["id"]: g["name"] for g in _tmdb_get("/genre/movie/list", language="en-US")["genres"]}
        rows = {}
        for src, path in (("In theaters", "/movie/now_playing"), ("Trending", "/trending/movie/week")):
            for page in (1, 2, 3):
                for m in _tmdb_get(path, language="en-US", page=page)["results"]:
                    if not m.get("overview") or not m.get("poster_path"):
                        continue
                    r = rows.setdefault(m["id"], dict(
                        id=m["id"], title=m["title"], year=_year(m.get("release_date")),
                        genres=", ".join(gmap[g] for g in m.get("genre_ids", []) if g in gmap),
                        overview=m["overview"], poster_path=m["poster_path"],
                        vote_average=float(m.get("vote_average") or 0), sources=[]))
                    if src not in r["sources"]:
                        r["sources"].append(src)
        df = pd.DataFrame(list(rows.values()))
        texts = (df["title"] + ". Genres: " + df["genres"] + ". " + df["overview"]).tolist()
        e = _get_model().encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        _live.update(df=df, emb=e, t=time.time(), error=None)
    except Exception as ex:
        _live.update(df=None, emb=None, error=f"{type(ex).__name__}: {ex}")
    return _live["df"], _live["emb"]


# ---------------------------------------------------------------- recommending
def _mask(genres, min_year, max_runtime):
    m = cat["year"].values >= min_year
    rt = cat["runtime"].values
    m = m & ((rt <= max_runtime) | (rt <= 0))
    if genres:
        gs = set(genres)
        m = m & np.array([bool(gs & s) for s in _gsets])
    return m


def _why(i, idx_groups):
    if len(idx_groups) == 1:
        g = idx_groups[0]
        j = g[int(np.argmax(emb[g] @ emb[i]))]
        bits = [f"Because you liked <b>{esc(cat['title'][j])}</b>"]
        shared_g = sorted(_gsets[i] & _gsets[j])[:3]
        liked_kw = set().union(*[_ksets[k] for k in g])
        shared_k = sorted(_ksets[i] & liked_kw)[:3]
        if shared_g:
            bits.append("genres: " + ", ".join(shared_g))
        if shared_k:
            bits.append("themes: " + ", ".join(shared_k))
        return " · ".join(bits)
    names = []
    for who, g in zip("AB", idx_groups):
        j = g[int(np.argmax(emb[g] @ emb[i]))]
        names.append(f"{who}'s <b>{esc(cat['title'][j])}</b>")
    return "Bridges " + " and ".join(names)


def _item(i, why):
    return dict(title=cat["title"][i], year=int(cat["year"][i]), poster=cat["poster_path"][i],
                rating=float(cat["vote_average"][i]), why=why,
                overview=str(cat["overview"][i]), badge="")


def recommend(groups, n=10, genres=None, min_year=1900, max_runtime=400,
              pop_w=0.1, include_live=True, n_live=6, genre_w=0.2, watched=None, hide_franchise=True):
    """groups: [liked_labels] for one person, [liked_a, liked_b] for a pair.
    watched: labels to hide from the results (already seen).
    hide_franchise: also hide sequels / same-series movies of the picks.
    Returns (items, live_items, live_error)."""
    idx_groups = [[_label_to_idx[l] for l in g if l in _label_to_idx] for g in groups]
    if not idx_groups or any(len(g) == 0 for g in idx_groups):
        return [], [], None
    all_idx = sorted({i for g in idx_groups for i in g})

    scores = _combine([_person_scores(emb, _G, g, genre_w) for g in idx_groups]) + pop_w * pop
    mask = _mask(genres, min_year, max_runtime)
    watched_idx = [_label_to_idx[l] for l in (watched or []) if l in _label_to_idx]
    mask[all_idx] = False
    mask[watched_idx] = False
    blocked = set()
    if hide_franchise:
        blocked = {_fkeys[i] for i in all_idx}
        mask &= ~np.isin(_fkeys, list(blocked))
    scores = np.where(mask, scores, -np.inf)
    top = _select(scores, all_idx, n)
    items = [_item(i, _why(i, idx_groups)) for i in top]

    live_items, live_err = [], None
    if include_live:
        ldf, lemb = get_live()
        if ldf is None:
            live_err = _live["error"] or "unknown error"
        else:
            lG = _multihot(ldf["genres"])
            ls = _combine([_person_scores(lemb, lG, g, genre_w) for g in idx_groups])
            lm = ldf["year"].values >= min_year
            if genres:
                gs = set(genres)
                lm = lm & np.array([bool(gs & {x.strip() for x in s.split(",")}) for s in ldf["genres"]])
            seen_ids = set(cat["id"].values[all_idx + watched_idx])
            lm = lm & ~ldf["id"].isin(seen_ids).values
            if blocked:
                lm = lm & np.array([franchise_key(t) not in blocked for t in ldf["title"]])
            ls = np.where(lm, ls, -np.inf)
            for i in [i for i in np.argsort(-ls)[:n_live] if np.isfinite(ls[i])]:
                row = ldf.iloc[i]
                if len(idx_groups) == 1:
                    g = idx_groups[0]
                    j = g[int(np.argmax(emb[g] @ lemb[i]))]
                    why = f"Close to <b>{esc(cat['title'][j])}</b>"
                else:
                    why = "A fresh pick for the two of you"
                live_items.append(dict(
                    title=row["title"], year=int(row["year"]), poster=row["poster_path"],
                    rating=float(row["vote_average"]), why=why, overview=str(row["overview"]),
                    badge=" + ".join(row["sources"])))
    return items, live_items, live_err

In [ ]:
import importlib, recsys as R
importlib.reload(R)

print(R.find("inception")[:3])
liked = [R.find(q)[0] for q in ["Inception", "Interstellar", "The Matrix"]]
print("liked:", liked)

items, live_items, err = R.recommend([liked], n=8)
print("\nTop picks")
for it in items:
    print(f"  {it['title']} ({it['year']})  -  {it['why'].replace('<b>', '').replace('</b>', '')}")
print("\nIn theaters / trending now" if not err else f"\nLive data problem: {err}")
for it in live_items:
    print(f"  {it['title']} ({it['year']}) [{it['badge']}]  -  {it['why'].replace('<b>', '').replace('</b>', '')}")

## 4 · Does it actually work? (evaluation on MovieLens)
This is what turns a demo into a project HRs respect. We use real ratings from MovieLens users: for each user, the **first 70%** of the movies they rated ≥ 4 stars build their taste profile, and we check whether the recommender finds the **later 30%** they also liked.

Compared: a **popularity baseline** (just show the most popular movies), the first version (**average taste**), the improved version (**per-pick matching + genre bonus**), and blends with popularity. This is a mini before/after story for your README. (The app's extra diversity step isn't scored here.) Report whatever numbers you get, even if the baseline is hard to beat; honest results look better than suspiciously perfect ones.

In [ ]:
import io, zipfile, urllib.request

URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=60).read()))
ratings = pd.read_csv(zf.open("ml-latest-small/ratings.csv"))
links = pd.read_csv(zf.open("ml-latest-small/links.csv")).dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype(int)

id2row = {tid: i for i, tid in enumerate(R.cat["id"].values)}
links["row"] = links["tmdbId"].map(id2row)
liked_df = (ratings[ratings["rating"] >= 4.0]
            .merge(links[["movieId", "row"]], on="movieId")
            .dropna(subset=["row"]).sort_values(["userId", "timestamp"]))
liked_df["row"] = liked_df["row"].astype(int)
print(f"{len(liked_df):,} liked ratings matched to our catalog, from {liked_df['userId'].nunique()} users")

K = 10
MAX_YEAR = int(pd.to_datetime(ratings["timestamp"].max(), unit="s").year)   # MovieLens can't contain later releases
fair = R.cat["year"].values <= MAX_YEAR
print(f"Candidate pool: {fair.sum():,} of {len(fair):,} movies (released up to {MAX_YEAR})")

NAMES = ["Popularity baseline", "Content: average taste (v1)", "Content: per-pick + genre bonus (v2)",
         "v2 + popularity (0.25)", "v2 + popularity (0.50)", "v2 + popularity (1.00)"]
res = {n: {"Precision@10": [], "Recall@10": [], "NDCG@10": []} for n in NAMES}
recs = {n: [] for n in NAMES}          # every user's top-10 (for coverage / popularity stats)
n_users = 0

for uid, g in liked_df.groupby("userId"):
    rows = g["row"].tolist()
    if len(rows) < 15:
        continue
    cut = int(len(rows) * 0.7)
    profile, test = rows[:cut], set(rows[cut:])
    taste = R._unit(R.emb[profile].mean(0))
    sim = R.emb @ taste
    v2 = R._person_scores(R.emb, R._G, profile, 0.2)
    methods = {NAMES[0]: R.pop, NAMES[1]: sim, NAMES[2]: v2,
               NAMES[3]: v2 + 0.25 * R.pop, NAMES[4]: v2 + 0.50 * R.pop, NAMES[5]: v2 + 1.00 * R.pop}
    for name, sc in methods.items():
        sc = sc.astype("float64").copy()
        sc[~fair] = -np.inf                        # only movies that could exist in MovieLens
        sc[profile] = -np.inf                      # never recommend what they already liked
        top = np.argpartition(-sc, K)[:K]
        top = top[np.argsort(-sc[top])]
        hits = np.array([t in test for t in top], dtype=float)
        dcg = (hits / np.log2(np.arange(2, K + 2))).sum()
        idcg = (1 / np.log2(np.arange(2, min(len(test), K) + 2))).sum()
        res[name]["Precision@10"].append(hits.mean())
        res[name]["Recall@10"].append(hits.sum() / len(test))
        res[name]["NDCG@10"].append(dcg / idcg)
        recs[name].append(top)
    n_users += 1

out = {}
for name in NAMES:
    allrecs = np.concatenate(recs[name])
    out[name] = {**{m: np.mean(v) for m, v in res[name].items()},
                 "Coverage": len(set(allrecs.tolist())) / fair.sum(),   # share of the pool ever recommended
                 "Avg popularity": R.pop[allrecs].mean()}               # 0 = obscure, 1 = blockbuster
table = pd.DataFrame(out).T.round(4)
print(f"Evaluated on {n_users} users\n")
table

In [ ]:
# markdown version of the table, paste straight into your README
cols = list(table.columns)
lines = ["| Method | " + " | ".join(cols) + " |", "|---|" + "---|" * len(cols)]
for name, row in table.iterrows():
    lines.append(f"| {name} | " + " | ".join(f"{row[c]:.4f}" for c in cols) + " |")
print("\n".join(lines))
print(f"\n(evaluated on {n_users} MovieLens users, temporal 70/30 split of each user's liked movies)")

## 5 · The app (`app.py`)
Two tabs: **Just me** and **Movie night for two** (finds movies that work for both people), with filters and a live in-theaters / trending section.

In [ ]:
%%writefile app.py
"""Gradio UI for Movie Night Picker."""
import html

import gradio as gr

import recsys as R

STYLE = """<style>
.mn-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(310px,1fr));gap:14px;}
.mn-card{display:flex;gap:12px;padding:10px;border:1px solid var(--border-color-primary);
  border-radius:12px;background:var(--background-fill-secondary);}
.mn-card img{width:92px;height:138px;object-fit:cover;border-radius:8px;flex-shrink:0;background:#0003;}
.mn-t{font-weight:600;font-size:1rem;}
.mn-y{opacity:.65;font-weight:400;}
.mn-r{font-size:.85rem;margin:2px 0 6px;opacity:.85;}
.mn-w{font-size:.85rem;}
.mn-o{font-size:.78rem;opacity:.65;margin-top:6px;}
.mn-badge{background:var(--color-accent-soft);border-radius:6px;padding:1px 6px;font-size:.72rem;margin-left:6px;}
h3.mn-h{margin:18px 0 8px;}
</style>"""


def cards(items):
    out = []
    for it in items:
        img = f"<img src='{R.IMG}{it['poster']}' loading='lazy'>" if it.get("poster") else "<div style='width:92px'></div>"
        badge = f"<span class='mn-badge'>{html.escape(it['badge'])}</span>" if it.get("badge") else ""
        ov = it["overview"]
        ov = html.escape(ov[:150]) + ("…" if len(ov) > 150 else "")
        out.append(
            f"<div class='mn-card'>{img}<div>"
            f"<div class='mn-t'>{html.escape(it['title'])} <span class='mn-y'>({it['year']})</span></div>"
            f"<div class='mn-r'>★ {it['rating']:.1f}{badge}</div>"
            f"<div class='mn-w'>{it['why']}</div>"
            f"<div class='mn-o'>{ov}</div></div></div>")
    return "<div class='mn-grid'>" + "".join(out) + "</div>"


def page(items, live_items, live_err, include_live):
    if not items:
        return STYLE + "<p>No matches. Try loosening the filters or picking different movies.</p>"
    out = STYLE + "<h3 class='mn-h'>🎯 Top picks</h3>" + cards(items)
    if include_live:
        if live_err:
            out += f"<p>⚠️ Couldn't load live TMDB data ({html.escape(live_err)}).</p>"
        elif live_items:
            out += "<h3 class='mn-h'>🎟️ In theaters & trending now</h3>" + cards(live_items)
    return out


def solo(liked, n, genres, min_year, max_rt, pop_w, genre_w, live, watched, hide_fr):
    if not liked:
        return STYLE + "<p>👆 Pick a few movies you like first.</p>"
    res = R.recommend([liked], int(n), genres, int(min_year), int(max_rt), float(pop_w), bool(live), genre_w=float(genre_w),
                      watched=watched, hide_franchise=bool(hide_fr))
    return page(*res, live)


def duo(liked_a, liked_b, n, genres, min_year, max_rt, pop_w, genre_w, live, watched, hide_fr):
    if not liked_a or not liked_b:
        return STYLE + "<p>👆 Both of you need to pick at least one movie.</p>"
    res = R.recommend([liked_a, liked_b], int(n), genres, int(min_year), int(max_rt), float(pop_w), bool(live), genre_w=float(genre_w),
                      watched=watched, hide_franchise=bool(hide_fr))
    return page(*res, live)


CHOICES = R.picker_choices()

with gr.Blocks(title="Movie Night Picker") as demo:
    gr.Markdown("# 🍿 Movie Night Picker\nPick movies you love and get recommendations, "
                "including what's in theaters and trending right now. Works for one person or two.")
    with gr.Accordion("Filters", open=False):
        genres = gr.CheckboxGroup(R.ALL_GENRES, label="Only these genres (any of them)")
        with gr.Row():
            min_year = gr.Slider(1920, 2026, value=1980, step=1, label="Released since")
            max_rt = gr.Slider(60, 300, value=300, step=5, label="Max runtime (min)")
        with gr.Row():
            n = gr.Slider(5, 20, value=10, step=1, label="How many picks")
            pop_w = gr.Slider(0.0, 0.3, value=0.1, step=0.01,
                              label="Crowd-pleaser boost (0 = pure taste match)")
        genre_w = gr.Slider(0.0, 0.5, value=0.2, step=0.05,
                            label="Stay close to my genres (0 = plot similarity only)")
        live = gr.Checkbox(value=True, label="Also show movies in theaters / trending now")
        hide_fr = gr.Checkbox(value=True, label="Hide sequels / same series as my picks")
        watched = gr.Dropdown(CHOICES, multiselect=True, filterable=True,
                              label="Already watched (never recommend these)")
    with gr.Tabs():
        with gr.Tab("Just me"):
            liked = gr.Dropdown(CHOICES, multiselect=True, filterable=True, max_choices=8,
                                label="Movies you like (type to search, pick 1-8)")
            go1 = gr.Button("Find my movie", variant="primary")
            out1 = gr.HTML()
            go1.click(solo, [liked, n, genres, min_year, max_rt, pop_w, genre_w, live, watched, hide_fr], out1)
        with gr.Tab("Movie night for two"):
            with gr.Row():
                a = gr.Dropdown(CHOICES, multiselect=True, filterable=True, max_choices=8, label="Person A likes")
                b = gr.Dropdown(CHOICES, multiselect=True, filterable=True, max_choices=8, label="Person B likes")
            go2 = gr.Button("Find something for both of us", variant="primary")
            out2 = gr.HTML()
            go2.click(duo, [a, b, n, genres, min_year, max_rt, pop_w, genre_w, live, watched, hide_fr], out2)
    gr.Markdown("<small>Movie data from TMDB. This product uses the TMDB API but is not endorsed "
                "or certified by TMDB.</small>")

if __name__ == "__main__":
    demo.launch()

In [ ]:
import importlib, sys
sys.path.insert(0, ".")
import recsys, app
importlib.reload(recsys); importlib.reload(app)

# share=True also gives you a temporary public link (valid ~1 week) to send to friends
app.demo.launch(share=True, debug=False)   # debug=False so the cells below can still run

## 6 · Build the website (free hosting on GitHub Pages)
The Gradio app above is great for Colab, but its link dies when Colab stops, and Hugging Face now charges for Gradio Spaces. So we also build a **static website** (HTML + JavaScript) that runs the same recommender **in the visitor's browser**. It's free to host forever, and anyone can use it with no install.

Cells below: write the website files, export the movie data (the embeddings become a small binary file), then download everything as a zip.

In [ ]:
!mkdir -p web/data

In [ ]:
%%writefile web/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Movie Night Picker</title>
<meta name="description" content="Pick movies you love, get recommendations for tonight. Works for one person or two.">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<link rel="stylesheet" href="style.css">
</head>
<body>
<div class="curtain"></div>
<header class="marquee">
  <div class="bulbs top"></div>
  <p class="now">★ NOW SHOWING ★</p>
  <h1>Movie Night Picker</h1>
  <p class="tag">Tell me what you love. I'll find tonight's movie.</p>
  <div class="bulbs bottom"></div>
</header>

<main class="wrap">
  <section class="panel" id="picks">
    <div class="row between">
      <h2>🎟️ Your picks</h2>
      <label class="switch"><input type="checkbox" id="two"> Movie night for two 👫</label>
    </div>
    <div class="pickers">
      <div class="pcol">
        <label class="lbl" id="lblA">Movies you love <span class="hint">(pick 1–8)</span></label>
        <div class="picker" id="pickA" data-key="likedA">
          <div class="chips"></div>
          <input type="text" placeholder="Type a title, e.g. Inception…" autocomplete="off" aria-label="Search movies you love">
          <ul class="sugg" hidden></ul>
        </div>
      </div>
      <div class="pcol" id="colB" hidden>
        <label class="lbl">Person B loves <span class="hint">(pick 1–8)</span></label>
        <div class="picker" id="pickB" data-key="likedB">
          <div class="chips"></div>
          <input type="text" placeholder="Type a title…" autocomplete="off" aria-label="Search movies person B loves">
          <ul class="sugg" hidden></ul>
        </div>
      </div>
    </div>

    <details class="tune">
      <summary>🎛️ Fine-tune</summary>
      <div class="tune-grid">
        <div class="full">
          <span class="lbl">Only these genres <span class="hint">(any of them; none selected = all)</span></span>
          <div id="genreChips" class="gchips"></div>
        </div>
        <label>Released since <output id="oYear">1980</output>
          <input type="range" id="minYear" min="1920" max="2026" value="1980"></label>
        <label>Max runtime <output id="oRt">any</output>
          <input type="range" id="maxRt" min="60" max="300" step="5" value="300"></label>
        <label>How many picks <output id="oN">12</output>
          <input type="range" id="nPicks" min="6" max="24" step="2" value="12"></label>
        <label>Crowd-pleaser boost <output id="oPop">0.10</output>
          <input type="range" id="popW" min="0" max="0.3" step="0.01" value="0.1"></label>
        <label>Stay close to my genres <output id="oGen">0.20</output>
          <input type="range" id="genreW" min="0" max="0.5" step="0.05" value="0.2"></label>
        <label class="check"><input type="checkbox" id="hideFr" checked> Hide sequels / same series as my picks</label>
      </div>
    </details>

    <div class="row">
      <button class="btn" id="go">🎬 Roll film</button>
      <button class="btn ghost" id="share" title="Copy a link with your picks">🔗 Copy link</button>
      <span id="status" class="status"></span>
    </div>
  </section>

  <section class="panel" id="watchedPanel">
    <h2>👀 Already watched <span class="hint">(never recommended; saved in this browser only)</span></h2>
    <div class="picker" id="pickW" data-key="watched">
      <div class="chips"></div>
      <input type="text" placeholder="Add movies you've seen…" autocomplete="off" aria-label="Search movies you have watched">
      <ul class="sugg" hidden></ul>
    </div>
  </section>

  <div class="strip"></div>
  <section id="results" aria-live="polite"></section>
  <div class="strip"></div>
</main>

<footer>
  <p>Movie data and posters from TMDB. This product uses the TMDB API but is not endorsed or certified by TMDB.</p>
  <p>Recommendations run entirely in your browser (text embeddings of plots, genres &amp; keywords). Code on <a href="https://github.com/YOUR_USERNAME/movie-night-picker">GitHub</a>.</p>
</footer>
<script src="app.js"></script>
</body>
</html>

In [ ]:
%%writefile web/style.css
:root{--bg:#0b0608;--panel:#150e11;--gold:#ffd27a;--red:#c8102e;--velvet:#5a0a18;--text:#f4ece6;--muted:#b9a99f;color-scheme:dark}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;min-height:100vh;color:var(--text);font-family:'Inter',system-ui,-apple-system,'Segoe UI',sans-serif;
  background:radial-gradient(1200px 520px at 50% -8%,#3d0b16 0,transparent 70%),var(--bg)}
body::before{content:"";position:fixed;inset:0;pointer-events:none;opacity:.07;z-index:0;
  background-image:url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='160' height='160'><filter id='n'><feTurbulence type='fractalNoise' baseFrequency='.9' numOctaves='2'/></filter><rect width='100%' height='100%' filter='url(%23n)'/></svg>")}
.curtain{position:relative;height:74px;z-index:1;
  background:repeating-linear-gradient(90deg,#430812 0 34px,#7a1124 34px 68px,#430812 68px 100px);
  box-shadow:inset 0 -22px 26px rgba(0,0,0,.55),0 10px 30px rgba(0,0,0,.65)}
.curtain::after{content:"";position:absolute;left:0;right:0;bottom:-12px;height:14px;
  background:radial-gradient(circle at 50% 0,#7a1124 0 15px,transparent 16px) 0 0/50px 14px repeat-x}
.marquee{position:relative;z-index:2;max-width:780px;margin:-26px auto 0;padding:26px 28px 24px;text-align:center;
  background:linear-gradient(#1b0e11,#0f0709);border:3px solid var(--gold);border-radius:14px;
  box-shadow:0 0 34px rgba(255,190,90,.28),inset 0 0 40px rgba(0,0,0,.6)}
.marquee h1{margin:2px 0 4px;font-family:'Bebas Neue',Impact,'Arial Narrow',sans-serif;font-weight:400;
  font-size:clamp(2.4rem,7vw,4rem);letter-spacing:.07em;color:#fff3d6;
  text-shadow:0 0 8px rgba(255,200,110,.9),0 0 26px rgba(255,150,60,.55)}
.marquee .now{margin:0;letter-spacing:.3em;font-size:.72rem;color:var(--gold)}
.marquee .tag{margin:0;color:var(--muted);font-size:.95rem}
.bulbs{position:absolute;left:12px;right:12px;height:8px;
  background:radial-gradient(circle,#ffe6a8 0 3px,transparent 4px) 0 50%/22px 8px repeat-x;animation:blink 1.6s steps(1) infinite}
.bulbs.top{top:6px}.bulbs.bottom{bottom:6px;animation-delay:.8s}
@keyframes blink{0%{opacity:1}50%{opacity:.35}100%{opacity:1}}
@media (prefers-reduced-motion:reduce){.bulbs{animation:none}}

.wrap{position:relative;z-index:1;max-width:1100px;margin:30px auto 20px;padding:0 16px;display:flex;flex-direction:column;gap:18px}
.panel{background:var(--panel);border:1px solid rgba(255,210,122,.2);border-radius:16px;padding:18px 20px;
  box-shadow:0 10px 30px rgba(0,0,0,.35)}
h2{margin:0 0 10px;font-family:'Bebas Neue',Impact,sans-serif;font-weight:400;letter-spacing:.06em;font-size:1.5rem;color:var(--gold)}
.row{display:flex;align-items:center;gap:12px;flex-wrap:wrap;margin-top:12px}
.row.between{justify-content:space-between;margin-top:0}
.hint{color:var(--muted);font-size:.78rem;font-family:'Inter',sans-serif;letter-spacing:0}
.lbl{display:block;font-weight:600;margin-bottom:6px;font-size:.92rem}
.switch{cursor:pointer;font-size:.9rem;user-select:none}
.pickers{display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:16px}

.picker{position:relative;background:#0e080a;border:1px solid rgba(255,210,122,.25);border-radius:12px;padding:8px}
.picker:focus-within{border-color:var(--gold);box-shadow:0 0 0 3px rgba(255,210,122,.15)}
.chips{display:flex;flex-wrap:wrap;gap:6px;margin-bottom:6px}
.chip{display:inline-flex;align-items:center;gap:6px;background:#2a1318;border:1px solid rgba(255,210,122,.35);
  border-radius:999px;padding:3px 6px 3px 11px;font-size:.84rem}
.chip button{border:0;background:rgba(255,255,255,.12);color:var(--text);width:19px;height:19px;border-radius:50%;cursor:pointer;line-height:1}
.chip button:hover{background:var(--red)}
.picker input{width:100%;background:transparent;border:0;outline:0;color:var(--text);font-size:.95rem;padding:6px 4px}
.sugg{position:absolute;left:0;right:0;top:100%;margin:6px 0 0;padding:4px;list-style:none;z-index:20;
  background:#1a0f12;border:1px solid rgba(255,210,122,.4);border-radius:10px;max-height:300px;overflow:auto;box-shadow:0 14px 30px rgba(0,0,0,.6)}
.sugg li{padding:8px 10px;border-radius:7px;cursor:pointer;font-size:.9rem;display:flex;justify-content:space-between;gap:10px}
.sugg li:hover,.sugg li.on{background:#3a1a21}
.sugg li small{color:var(--muted);white-space:nowrap}

.tune{margin-top:14px}
.tune summary{cursor:pointer;color:var(--gold);font-weight:600}
.tune-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:14px 22px;margin-top:12px}
.tune-grid .full{grid-column:1/-1}
.tune-grid label{display:flex;flex-direction:column;gap:6px;font-size:.88rem}
.tune-grid label.check{flex-direction:row;align-items:center}
.tune-grid output{color:var(--gold);float:right}
input[type=range]{accent-color:var(--red);width:100%}
.gchips{display:flex;flex-wrap:wrap;gap:6px}
.g{border:1px solid rgba(255,210,122,.3);background:transparent;color:var(--text);border-radius:999px;padding:4px 11px;font-size:.8rem;cursor:pointer}
.g.on{background:var(--red);border-color:var(--red)}

.btn{background:linear-gradient(#e0132f,#a10a20);color:#fff;border:0;border-radius:10px;padding:11px 22px;font-weight:600;
  font-size:1rem;cursor:pointer;box-shadow:0 6px 18px rgba(200,16,46,.35)}
.btn:hover{filter:brightness(1.1)}
.btn.ghost{background:transparent;border:1px solid rgba(255,210,122,.4);box-shadow:none;color:var(--text)}
.status{color:var(--muted);font-size:.85rem}

.strip{height:20px;opacity:.9;
  background:linear-gradient(#0a0a0a,#0a0a0a);
  -webkit-mask:repeating-linear-gradient(90deg,#000 0 12px,transparent 12px 20px);mask:repeating-linear-gradient(90deg,#000 0 12px,transparent 12px 20px);
  border-top:3px solid #3a3330;border-bottom:3px solid #3a3330}
.section-title{display:flex;align-items:baseline;gap:12px;margin:8px 0 12px}
.section-title h2{margin:0;font-size:1.8rem}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(340px,1fr));gap:18px;margin-bottom:26px}
.card{position:relative;display:flex;background:linear-gradient(135deg,#1e1317,#130b0e);overflow:hidden;
  border:1px solid rgba(255,210,122,.25);border-radius:14px;box-shadow:0 8px 22px rgba(0,0,0,.45)}
.card::before,.card::after{content:"";position:absolute;left:103px;width:18px;height:18px;border-radius:50%;background:var(--bg);
  border:1px solid rgba(255,210,122,.25)}
.card::before{top:-10px}.card::after{bottom:-10px}
.poster{position:relative;width:112px;flex-shrink:0;background:#000}
.poster img{width:100%;height:100%;object-fit:cover;display:block}
.rank{position:absolute;top:6px;left:6px;background:var(--gold);color:#2a1408;font-weight:700;font-size:.72rem;border-radius:999px;padding:2px 8px}
.body{padding:12px 14px;border-left:2px dashed rgba(255,210,122,.3);display:flex;flex-direction:column;gap:5px;min-width:0}
.t{font-weight:600;font-size:1.02rem;line-height:1.25}
.t .y{color:var(--muted);font-weight:400}
.meta{font-size:.82rem;color:#e8d9c8}
.badge{background:rgba(255,210,122,.16);color:var(--gold);border-radius:6px;padding:1px 7px;font-size:.7rem;margin-left:6px}
.why{font-size:.84rem}
.ov{font-size:.76rem;color:var(--muted)}
.acts{margin-top:auto;display:flex;gap:6px;padding-top:6px}
.acts button{background:transparent;border:1px solid rgba(255,210,122,.35);color:var(--text);border-radius:8px;padding:4px 9px;font-size:.76rem;cursor:pointer}
.acts button:hover{background:#3a1a21}
.empty{padding:34px 10px;text-align:center;color:var(--muted)}
.asof{color:var(--muted);font-size:.8rem}
.loading{text-align:center;padding:40px;color:var(--muted)}
footer{position:relative;z-index:1;text-align:center;color:var(--muted);font-size:.78rem;padding:10px 16px 40px}
footer a{color:var(--gold)}
@media (max-width:420px){.card::before,.card::after{display:none}.poster{width:96px}}

In [ ]:
%%writefile web/app.js
/* Movie Night Picker - runs fully in the browser.
   Data: data/movies.json (movie columns), data/emb.bin (int8 text embeddings), data/live.json (in-theaters snapshot). */
(() => {
'use strict';

const IMG = 'https://image.tmdb.org/t/p/w342';
const MAX_PICKS = 8;
const LS_WATCHED = 'mnp_watched_v1';

let M, EMB, N = 0, DIM = 0, INV = 1;
let GENRES = [], GN = 0, GH, GC, POP, TLC = [], ID2ROW = new Map();
let LIVE = null;
const kwCache = new Map();

const S = { likedA: [], likedB: [], watched: [], two: false, genres: new Set(),
            minYear: 1980, maxRt: 300, popW: 0.1, genreW: 0.2, hideFr: true, n: 12 };

const $ = (s, r = document) => r.querySelector(s);
const esc = s => String(s).replace(/[&<>"']/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;' }[c]));
const safeLS = {
  get(k) { try { return localStorage.getItem(k); } catch (e) { return null; } },
  set(k, v) { try { localStorage.setItem(k, v); } catch (e) { /* storage can be blocked */ } }
};

/* ------------------------------------------------------------------ data */
function multihot(genreStrings) {
  const gi = new Map(GENRES.map((g, i) => [g, i]));
  const n = genreStrings.length, H = new Uint8Array(n * GN), C = new Uint8Array(n);
  for (let r = 0; r < n; r++) {
    for (const g of String(genreStrings[r] || '').split(',')) {
      const k = gi.get(g.trim());
      if (k !== undefined && !H[r * GN + k]) { H[r * GN + k] = 1; C[r]++; }
    }
  }
  return [H, C];
}

function b64ToInt8(b64) {
  const bin = atob(b64), out = new Int8Array(bin.length);
  for (let i = 0; i < bin.length; i++) out[i] = (bin.charCodeAt(i) << 24) >> 24;
  return out;
}

async function loadData() {
  const [meta, buf, live] = await Promise.all([
    fetch('data/movies.json').then(r => r.json()),
    fetch('data/emb.bin').then(r => r.arrayBuffer()),
    fetch('data/live.json').then(r => r.ok ? r.json() : null).catch(() => null)
  ]);
  M = meta.cols; N = meta.n; DIM = meta.dim; INV = 1 / (meta.scale * meta.scale);
  GENRES = meta.genres; GN = GENRES.length;
  EMB = new Int8Array(buf);
  if (EMB.length !== N * DIM) throw new Error('embeddings and movie table are out of sync');
  [GH, GC] = multihot(M.genres);
  POP = new Float32Array(N);
  let lo = Infinity, hi = -Infinity;
  for (let i = 0; i < N; i++) { POP[i] = Math.log1p(M.votes[i]); lo = Math.min(lo, POP[i]); hi = Math.max(hi, POP[i]); }
  for (let i = 0; i < N; i++) POP[i] = (POP[i] - lo) / (hi - lo + 1e-9);
  TLC = M.title.map(t => t.toLowerCase());
  M.id.forEach((id, i) => ID2ROW.set(id, i));
  if (live && live.movies && live.movies.length) {
    const [H, C] = multihot(live.movies.map(m => m.genres));
    LIVE = { ...live, emb: b64ToInt8(live.emb), H, C, n: live.movies.length };
    if (LIVE.emb.length !== LIVE.n * DIM) LIVE = null;
  }
}

const kws = i => {
  let s = kwCache.get(i);
  if (!s) { s = new Set(M.kw[i] ? M.kw[i].split('|') : []); kwCache.set(i, s); }
  return s;
};

/* ------------------------------------------------------------------ maths */
function dotCat(i, j) {
  let d = 0; const a = i * DIM, b = j * DIM;
  for (let t = 0; t < DIM; t++) d += EMB[a + t] * EMB[b + t];
  return d * INV;
}

// score every row of E (catalog or live) for ONE person whose picks are catalog rows `picks`
function personScores(E, H, C, rows, picks, genreW) {
  const m = picks.length, out = new Float32Array(rows);
  const w = new Float32Array(GN);
  for (const p of picks) for (let g = 0; g < GN; g++) w[g] += GH[p * GN + g] / m;
  for (let i = 0; i < rows; i++) {
    let sum = 0, mx = -Infinity; const bi = i * DIM;
    for (let k = 0; k < m; k++) {
      const pj = picks[k] * DIM; let d = 0;
      for (let t = 0; t < DIM; t++) d += E[bi + t] * EMB[pj + t];
      d *= INV; sum += d; if (d > mx) mx = d;
    }
    let s = 0.5 * (sum / m) + 0.5 * mx;
    if (genreW) {
      let gs = 0; const gb = i * GN;
      for (let g = 0; g < GN; g++) if (H[gb + g]) gs += w[g];
      s += genreW * gs / Math.max(C[i], 1);
    }
    out[i] = s;
  }
  return out;
}

function combine(list) {
  if (list.length === 1) return list[0];
  const a = list[0], b = list[1], out = new Float32Array(a.length);
  for (let i = 0; i < a.length; i++) out[i] = 0.5 * (a[i] + b[i]) + 0.5 * Math.min(a[i], b[i]);
  return out;
}

// greedy choice from the top `pool`: score - similarity to already chosen - overuse of one pick as the reason
function select(scores, picks, n, lam = 0.15, quotaPen = 0.04, pool = 100) {
  const order = [];
  for (let i = 0; i < scores.length; i++) if (scores[i] > -Infinity) order.push(i);
  order.sort((a, b) => scores[b] - scores[a]);
  const cand = order.slice(0, pool);
  const near = new Map();
  for (const i of cand) {
    let best = 0, bd = -Infinity;
    picks.forEach((p, k) => { const d = dotCat(i, p); if (d > bd) { bd = d; best = k; } });
    near.set(i, best);
  }
  const used = new Map(), chosen = [], remaining = cand.slice();
  while (remaining.length && chosen.length < n) {
    let bestI = -1, bestV = -Infinity;
    for (const i of remaining) {
      let v = scores[i] - quotaPen * (used.get(near.get(i)) || 0);
      if (chosen.length) {
        let mx = -Infinity;
        for (const c of chosen) { const d = dotCat(i, c); if (d > mx) mx = d; }
        v -= lam * mx;
      }
      if (v > bestV) { bestV = v; bestI = i; }
    }
    chosen.push(bestI);
    remaining.splice(remaining.indexOf(bestI), 1);
    used.set(near.get(bestI), (used.get(near.get(bestI)) || 0) + 1);
  }
  return chosen;
}

/* ------------------------------------------------------------------ explanations */
function nearestPick(vecFn, picks) {
  let best = picks[0], bd = -Infinity;
  for (const p of picks) { const d = vecFn(p); if (d > bd) { bd = d; best = p; } }
  return best;
}

function sharedGenres(i, j) {
  const out = [];
  for (let g = 0; g < GN; g++) if (GH[i * GN + g] && GH[j * GN + g]) out.push(GENRES[g]);
  return out.slice(0, 3);
}

function whyCatalog(i, groups) {
  if (groups.length === 1) {
    const g = groups[0], j = nearestPick(p => dotCat(i, p), g);
    const bits = [`Because you liked <b>${esc(M.title[j])}</b>`];
    const sg = sharedGenres(i, j);
    const liked = new Set(); g.forEach(p => kws(p).forEach(k => liked.add(k)));
    const sk = [...kws(i)].filter(k => liked.has(k)).sort().slice(0, 3);
    if (sg.length) bits.push('genres: ' + sg.join(', '));
    if (sk.length) bits.push('themes: ' + sk.join(', '));
    return bits.join(' · ');
  }
  const names = groups.map((g, k) => `${'AB'[k]}'s <b>${esc(M.title[nearestPick(p => dotCat(i, p), g)])}</b>`);
  return 'Bridges ' + names.join(' and ');
}

function dotLiveCat(li, p) {
  let d = 0; const a = li * DIM, b = p * DIM;
  for (let t = 0; t < DIM; t++) d += LIVE.emb[a + t] * EMB[b + t];
  return d * INV;
}

/* ------------------------------------------------------------------ recommend */
function franchiseBlockedSet(allPicks) { return new Set(allPicks.map(i => M.fk[i])); }

function recommend() {
  const groups = S.two ? [S.likedA, S.likedB] : [S.likedA];
  if (groups.some(g => g.length === 0)) return null;
  const all = [...new Set(groups.flat())];
  const blocked = S.hideFr ? franchiseBlockedSet(all) : new Set();
  const watched = new Set(S.watched);
  const picked = new Set(all);

  const scores = combine(groups.map(g => personScores(EMB, GH, GC, N, g, S.genreW)));
  const pop = S.popW;
  for (let i = 0; i < N; i++) {
    let ok = M.year[i] >= S.minYear && (S.maxRt >= 300 || M.runtime[i] <= S.maxRt || M.runtime[i] <= 0);
    if (ok && S.genres.size) {
      ok = false;
      for (const g of S.genres) { const k = GENRES.indexOf(g); if (k >= 0 && GH[i * GN + k]) { ok = true; break; } }
    }
    if (ok && (picked.has(i) || watched.has(i) || blocked.has(M.fk[i]))) ok = false;
    scores[i] = ok ? scores[i] + pop * POP[i] : -Infinity;
  }
  const items = select(scores, all, S.n).map(i => ({
    row: i, title: M.title[i], year: M.year[i], poster: M.poster[i], rating: M.rating[i],
    why: whyCatalog(i, groups), overview: M.overview[i], badge: ''
  }));

  let live = [];
  if (LIVE) {
    const ls = combine(groups.map(g => personScores(LIVE.emb, LIVE.H, LIVE.C, LIVE.n, g, S.genreW)));
    const seenIds = new Set([...all, ...S.watched].map(i => M.id[i]));
    const order = [];
    for (let i = 0; i < LIVE.n; i++) {
      const m = LIVE.movies[i];
      let ok = m.year >= S.minYear && !seenIds.has(m.id) && !blocked.has(m.fk);
      if (ok && S.genres.size) ok = m.genres.split(',').some(g => S.genres.has(g.trim()));
      if (ok) order.push(i);
    }
    order.sort((a, b) => ls[b] - ls[a]);
    live = order.slice(0, 6).map(i => {
      const m = LIVE.movies[i];
      const why = groups.length === 1
        ? `Close to <b>${esc(M.title[nearestPick(p => dotLiveCat(i, p), groups[0])])}</b>`
        : 'A fresh pick for the two of you';
      return { row: -1, title: m.title, year: m.year, poster: m.poster, rating: m.rating, why,
               overview: m.overview, badge: m.sources.join(' + ') };
    });
  }
  return { items, live };
}

/* ------------------------------------------------------------------ rendering */
function card(it, rank, canAct) {
  const ov = it.overview.length > 150 ? esc(it.overview.slice(0, 150)) + '…' : esc(it.overview);
  const img = it.poster ? `<img src="${IMG}${esc(it.poster)}" alt="" loading="lazy" onerror="this.style.display='none'">` : '';
  const acts = canAct
    ? `<div class="acts"><button data-act="seen" data-row="${it.row}">✓ Seen it</button><button data-act="like" data-row="${it.row}">＋ I love it</button></div>` : '';
  return `<article class="card"><div class="poster">${img}<span class="rank">No. ${rank}</span></div>
    <div class="body"><div class="t">${esc(it.title)} <span class="y">(${it.year || '?'})</span></div>
    <div class="meta">★ ${Number(it.rating).toFixed(1)}${it.badge ? `<span class="badge">${esc(it.badge)}</span>` : ''}</div>
    <div class="why">${it.why}</div><div class="ov">${ov}</div>${acts}</div></article>`;
}

function render() {
  const out = $('#results');
  const groups = S.two ? [S.likedA, S.likedB] : [S.likedA];
  if (groups.some(g => g.length === 0)) {
    out.innerHTML = `<div class="empty">🍿 ${S.two ? 'Both of you need to pick at least one movie.' : 'Pick a movie you love above to get started.'}</div>`;
    return;
  }
  const r = recommend();
  if (!r.items.length) { out.innerHTML = '<div class="empty">No matches. Try loosening the filters.</div>'; return; }
  let html = `<div class="section-title"><h2>🎯 Top picks</h2></div><div class="grid">${r.items.map((it, k) => card(it, k + 1, true)).join('')}</div>`;
  if (r.live.length) {
    html += `<div class="section-title"><h2>🎟️ In theaters &amp; trending</h2><span class="asof">snapshot from ${esc(LIVE.asof)}</span></div>
      <div class="grid">${r.live.map((it, k) => card(it, k + 1, false)).join('')}</div>`;
  }
  out.innerHTML = html;
}

/* ------------------------------------------------------------------ pickers (autocomplete + chips) */
function renderChips(key) {
  const root = $(`.picker[data-key="${key}"] .chips`);
  root.innerHTML = S[key].map(r => `<span class="chip">${esc(M.title[r])} <small>${M.year[r]}</small><button data-key="${key}" data-row="${r}" aria-label="Remove">×</button></span>`).join('');
}

function addPick(key, row) {
  const limit = key === 'watched' ? 400 : MAX_PICKS;
  if (S[key].includes(row) || S[key].length >= limit) return;
  S[key].push(row); renderChips(key); changed();
}

function initPicker(el) {
  const key = el.dataset.key, input = $('input', el), list = $('.sugg', el);
  let active = -1, shown = [];
  const hide = () => { list.hidden = true; active = -1; };
  const show = () => {
    const q = input.value.trim().toLowerCase();
    if (q.length < 2) return hide();
    shown = [];
    for (let i = 0; i < N && shown.length < 8; i++) if (TLC[i].includes(q) && !S[key].includes(i)) shown.push(i);
    list.innerHTML = shown.map((r, k) => `<li data-row="${r}" class="${k === active ? 'on' : ''}">${esc(M.title[r])}<small>${M.year[r]}</small></li>`).join('')
      || '<li style="cursor:default">No matches</li>';
    list.hidden = false;
  };
  input.addEventListener('input', () => { active = -1; show(); });
  input.addEventListener('keydown', e => {
    if (e.key === 'ArrowDown' && shown.length) { active = (active + 1) % shown.length; show(); e.preventDefault(); }
    else if (e.key === 'ArrowUp' && shown.length) { active = (active - 1 + shown.length) % shown.length; show(); e.preventDefault(); }
    else if (e.key === 'Enter' && shown.length) { addPick(key, shown[Math.max(active, 0)]); input.value = ''; hide(); e.preventDefault(); }
    else if (e.key === 'Escape') hide();
    else if (e.key === 'Backspace' && !input.value && S[key].length) { S[key].pop(); renderChips(key); changed(); }
  });
  list.addEventListener('mousedown', e => {
    const li = e.target.closest('li[data-row]');
    if (li) { addPick(key, +li.dataset.row); input.value = ''; hide(); e.preventDefault(); }
  });
  input.addEventListener('blur', () => setTimeout(hide, 120));
  el.addEventListener('click', e => {
    const b = e.target.closest('.chip button');
    if (b) { S[key] = S[key].filter(r => r !== +b.dataset.row); renderChips(key); changed(); }
  });
}

/* ------------------------------------------------------------------ state, url, storage */
let timer = null;
function changed() {
  persist();
  clearTimeout(timer);
  timer = setTimeout(render, 120);
}

function persist() {
  safeLS.set(LS_WATCHED, JSON.stringify(S.watched.map(r => M.id[r])));
  const ids = a => a.map(r => M.id[r]).join(',');
  const p = [];
  if (S.likedA.length) p.push('a=' + ids(S.likedA));
  if (S.two) { p.push('two=1'); if (S.likedB.length) p.push('b=' + ids(S.likedB)); }
  try { history.replaceState(null, '', p.length ? '#' + p.join('&') : location.pathname + location.search); } catch (e) { /* ignore */ }
}

function restore() {
  const toRows = s => (s || '').split(',').map(x => ID2ROW.get(+x)).filter(r => r !== undefined).slice(0, MAX_PICKS);
  try {
    const w = JSON.parse(safeLS.get(LS_WATCHED) || '[]');
    S.watched = w.map(id => ID2ROW.get(id)).filter(r => r !== undefined);
  } catch (e) { S.watched = []; }
  const h = new URLSearchParams(location.hash.replace(/^#/, ''));
  S.likedA = toRows(h.get('a'));
  S.likedB = toRows(h.get('b'));
  S.two = h.get('two') === '1';
}

/* ------------------------------------------------------------------ controls */
function initControls() {
  const bindRange = (id, outId, key, fmt, parse = Number) => {
    const el = $('#' + id), out = $('#' + outId);
    const upd = () => { S[key] = parse(el.value); out.textContent = fmt(S[key]); };
    el.addEventListener('input', () => { upd(); changed(); });
    upd();
  };
  bindRange('minYear', 'oYear', 'minYear', v => v);
  bindRange('maxRt', 'oRt', 'maxRt', v => v >= 300 ? 'any' : v + ' min');
  bindRange('nPicks', 'oN', 'n', v => v, v => parseInt(v, 10));
  bindRange('popW', 'oPop', 'popW', v => v.toFixed(2));
  bindRange('genreW', 'oGen', 'genreW', v => v.toFixed(2));
  $('#hideFr').addEventListener('change', e => { S.hideFr = e.target.checked; changed(); });

  const gc = $('#genreChips');
  gc.innerHTML = GENRES.map(g => `<button type="button" class="g" data-g="${esc(g)}">${esc(g)}</button>`).join('');
  gc.addEventListener('click', e => {
    const b = e.target.closest('.g'); if (!b) return;
    const g = b.dataset.g;
    S.genres.has(g) ? S.genres.delete(g) : S.genres.add(g);
    b.classList.toggle('on', S.genres.has(g)); changed();
  });

  const two = $('#two');
  const applyTwo = () => {
    $('#colB').hidden = !S.two;
    $('#lblA').firstChild.textContent = S.two ? 'Person A loves ' : 'Movies you love ';
  };
  two.checked = S.two; applyTwo();
  two.addEventListener('change', () => { S.two = two.checked; applyTwo(); changed(); });

  $('#go').addEventListener('click', () => { clearTimeout(timer); render(); $('#results').scrollIntoView({ behavior: 'smooth', block: 'start' }); });
  $('#share').addEventListener('click', async () => {
    persist();
    try { await navigator.clipboard.writeText(location.href); $('#status').textContent = 'Link copied ✓'; }
    catch (e) { window.prompt('Copy this link:', location.href); }
    setTimeout(() => { $('#status').textContent = ''; }, 2500);
  });

  $('#results').addEventListener('click', e => {
    const b = e.target.closest('button[data-act]'); if (!b) return;
    const row = +b.dataset.row;
    if (b.dataset.act === 'seen') {
      if (!S.watched.includes(row)) S.watched.push(row);
      S.likedA = S.likedA.filter(r => r !== row); S.likedB = S.likedB.filter(r => r !== row);
      renderChips('watched'); renderChips('likedA'); renderChips('likedB'); changed();
    } else if (b.dataset.act === 'like') {
      addPick(S.two && S.likedA.length > 0 && S.likedB.length === 0 ? 'likedB' : 'likedA', row);
    }
  });
}

/* ------------------------------------------------------------------ boot */
async function boot() {
  const results = $('#results');
  results.innerHTML = '<div class="loading">🎞️ Loading the film library…</div>';
  try {
    await loadData();
  } catch (e) {
    results.innerHTML = `<div class="empty">Couldn't load the movie data (${esc(e.message)}).<br>If you opened this file directly, serve the folder instead, e.g. <code>python -m http.server</code>.</div>`;
    return;
  }
  restore();
  initControls();
  document.querySelectorAll('.picker').forEach(initPicker);
  ['likedA', 'likedB', 'watched'].forEach(renderChips);
  render();
}

if (typeof document !== 'undefined') boot();
})();

**Export the data.** The site can't hold your TMDB key safely (anyone could read it in the page), so the "in theaters / trending" section is a **snapshot** taken now. Re-run this cell and re-upload the files whenever you want it refreshed.

In [ ]:
import json, base64, datetime, importlib
import numpy as np
import recsys as R
importlib.reload(R)

N_WEB = 12000                      # movies shipped to the website (most popular first); smaller = faster page load
os.makedirs("web/data", exist_ok=True)

sub = R.cat.head(N_WEB)
E = R.emb[:len(sub)]
scale = 127.0 / float(np.abs(E).max())                       # int8 quantisation: 4x smaller than float32
q = np.clip(np.round(E * scale), -127, 127).astype("int8")
q.tofile("web/data/emb.bin")

# sanity check: do the quantised vectors still give the same similarities?
a = q[:300].astype("float32")
err = np.abs(a @ a.T / scale**2 - E[:300] @ E[:300].T).max()
print(f"max similarity error from int8 quantisation: {err:.4f} (should be ~0.01 or less)")

cols = {
    "id": sub["id"].astype(int).tolist(),
    "title": sub["title"].tolist(),
    "year": sub["year"].astype(int).tolist(),
    "genres": sub["genres"].tolist(),
    "kw": ["|".join(sorted(R._ksets[i])[:10]) for i in range(len(sub))],
    "poster": sub["poster_path"].tolist(),
    "rating": [round(float(x), 1) for x in sub["vote_average"]],
    "votes": sub["vote_count"].astype(int).tolist(),
    "runtime": sub["runtime"].astype(int).tolist(),
    "overview": [str(o)[:170] for o in sub["overview"]],
    "fk": sub["fkey"].tolist(),
}
meta = {"n": len(sub), "dim": int(E.shape[1]), "scale": scale,
        "built": str(datetime.date.today()), "genres": R.ALL_GENRES, "cols": cols}
with open("web/data/movies.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, separators=(",", ":"))

# snapshot of "in theaters / trending now" (static site => frozen at export time; re-run this cell to refresh)
ldf, lemb = R.get_live()
if ldf is None:
    print("⚠️ live movies unavailable:", R._live["error"], "-> site will work without that section")
    if os.path.exists("web/data/live.json"): os.remove("web/data/live.json")
else:
    ql = np.clip(np.round(lemb * scale), -127, 127).astype("int8")
    live = {"asof": datetime.date.today().strftime("%b %d, %Y"),
            "emb": base64.b64encode(ql.tobytes()).decode(),
            "movies": [{"id": int(r.id), "title": r.title, "year": int(r.year), "genres": r.genres,
                        "poster": r.poster_path, "rating": round(float(r.vote_average), 1),
                        "overview": str(r.overview)[:170], "sources": list(r.sources),
                        "fk": R.franchise_key(r.title)} for r in ldf.itertuples()]}
    json.dump(live, open("web/data/live.json", "w", encoding="utf-8"), ensure_ascii=False, separators=(",", ":"))
    print(f"live snapshot: {len(ldf)} movies")

for f in sorted(os.listdir("web/data")):
    print(f"web/data/{f}: {os.path.getsize('web/data/'+f)/1e6:.1f} MB")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("movie_night_site", "zip", "web")
files.download("movie_night_site.zip")

## 7 · Put it online (GitHub Pages)
1. On GitHub: **New repository** → name it `movie-night-picker` → Public.
2. Unzip the download. Upload the **contents** (`index.html`, `style.css`, `app.js` and the `data` folder) to the repo root (*Add file → Upload files*, drag them in).
3. Open `index.html` on GitHub, click the pencil icon, and replace `YOUR_USERNAME` in the footer link with your GitHub username.
4. Repo **Settings → Pages → Build and deployment → Deploy from a branch → `main` / `(root)` → Save**.
5. After ~1 minute your site is live at `https://YOUR_USERNAME.github.io/movie-night-picker/`.

Test locally first if you like: unzip, open a terminal in the folder, run `python -m http.server`, and visit `localhost:8000`. (Opening `index.html` by double-click won't work because browsers block loading the data files.)

**Before you publish**
- [ ] Add a screenshot or GIF of the site to the README, plus the results table from section 4 and 2-3 sentences on what you learned
- [ ] Keep the TMDB attribution line (already in the footer)
- [ ] Upload this notebook to the repo too (e.g. in a `notebook/` folder), but don't commit your keys. They only live in Colab Secrets
- [ ] Add the **Open in Colab** badge to your README

Ideas for v2: a TMDB "collection" field for exact franchise detection, a second embedding model compared in the results table, TV shows, or a "surprise me" button.